# OLS Regression Model — HDB Resale Price Prediction

Ordinary Least Squares baseline model predicting Singapore HDB resale prices using the enriched pipeline dataset (`hdb_with_amenities_macro.csv`).

**Target variable:** `resale_price_real` (RPI-adjusted, Q4 2025 SGD base = 203.6)  
**Period:** 2021–2025 (post-COVID; 2020 excluded due to structural break)  
**Final dataset:** ~96,800 transactions after cleaning

| Step | Description |
|------|-------------|
| 0 | Data exploration — flat type and flat model counts |
| 1 | Load and clean data |
| 2 | Feature engineering |
| 3 | Stratified train/test split (80/20) |
| 4 | OLS model fitting with VIF check |
| 5 | Diagnostics |
| 6 | Evaluation |

## Step 0: Data Exploration

Count of transactions by flat type and flat model from 2021 Q1 onwards.
Used to identify rare categories (`1 ROOM`: 41, `MULTI-GENERATION`: 43) that will be excluded in Step 1.

In [9]:
import pandas as pd

df = pd.read_csv("../../merged_data/hdb_with_amenities_macro.csv")
df_filtered = df[df["month"] >= "2021-01-01"]

flat_type_counts = df_filtered["flat_type"].value_counts()
print(f"Total records from 2021 Q1 onwards: {len(df_filtered):,}")
print("\nCount by flat type:")
print(flat_type_counts.to_string())

flat_model_counts = df_filtered["flat_model"].value_counts()
print(f"Total records from 2021 Q1 onwards: {len(df_filtered):,}")
print("\nCount by flat model:")
print(flat_model_counts.to_string())

Total records from 2021 Q1 onwards: 134,488

Count by flat type:
flat_type
4 ROOM              57847
5 ROOM              32611
3 ROOM              31902
EXECUTIVE            8885
2 ROOM               3159
MULTI-GENERATION       43
1 ROOM                 41
Total records from 2021 Q1 onwards: 134,488

Count by flat model:
flat_model
Model A                   50773
Improved                  32325
New Generation            15280
Premium Apartment         14946
Simplified                 4897
Apartment                  4415
Standard                   3486
Maisonette                 3479
DBSS                       2056
Model A2                   1413
2-room                      361
Model A-Maisonette          241
Type S1                     219
Adjoined flat               210
Type S2                     105
Premium Apartment Loft       88
3Gen                         66
Terrace                      61
Multi Generation             43
Improved-Maisonette          15
Premium Maisonette        

## Step 1: Load and Clean Data

Three sequential cleaning operations:

1. **Drop nulls** — rows with any missing value are removed (primarily affects amenity distance columns where geocoding returned no result: 157,821 → 114,147, dropped 43,674)
2. **Drop rare flat types** — `1 ROOM` (41 rows) and `MULTI-GENERATION` (43 rows) excluded due to insufficient sample size (< 50 each, < 0.1% of data); outside the typical buyer use case (114,147 → 114,057, dropped 90)
3. **Drop 2020** — excluded due to the COVID-19 structural break; abnormal market conditions unrepresentative of typical pricing relationships (114,057 → 96,796, dropped 17,261)

In [1]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("../../merged_data/hdb_with_amenities_macro.csv")
print(f"Initial shape: {df_raw.shape}")

df = df_raw.dropna()
print(f"After dropping nulls: {df.shape} (dropped {len(df_raw) - len(df)})")

# Excluded due to insufficient sample size (fewer than 50 transactions each),
# representing less than 0.1% of the dataset and falling outside the typical buyer use case.
mask_flat = ~df["flat_type"].isin(["1 ROOM", "MULTI-GENERATION"])
n_before = len(df)
df = df[mask_flat]
print(f"After dropping 1 ROOM and MULTI-GENERATION: {df.shape} (dropped {n_before - len(df)})")

# 2020 is excluded due to the COVID-19 structural break which caused abnormal market conditions
# unrepresentative of typical pricing relationships.
df["year_temp"] = pd.to_datetime(df["month"]).dt.year
n_before = len(df)
df = df[df["year_temp"] != 2020].drop(columns="year_temp")
print(f"After dropping 2020: {df.shape} (dropped {n_before - len(df)})")

df['log_resale_price_real'] = np.log(df['resale_price_real'])  # Log-transform target — preserves resale_price_real for metric computation

# Count of primary schools within 1 km — derived from pipe-separated names already in the dataset.
# Captures school density independently of nearest-school distance.
df['num_primary_1km'] = df['primary_schools_1km'].apply(
    lambda x: len(x.split('|')) if pd.notna(x) and x != '' else 0
)
print(f"num_primary_1km — mean: {df['num_primary_1km'].mean():.2f}, max: {df['num_primary_1km'].max()}")

Initial shape: (157821, 37)
After dropping nulls: (114147, 37) (dropped 43674)
After dropping 1 ROOM and MULTI-GENERATION: (114057, 37) (dropped 90)
After dropping 2020: (96796, 37) (dropped 17261)
num_primary_1km — mean: 2.99, max: 9


## Step 2: Feature Engineering

Three new columns created from existing raw columns:

| New column | Source | Logic |
|-----------|--------|-------|
| `remaining_lease_years` | `remaining_lease` | Parse `"61 years 04 months"` → `61.33` (handles missing months) |
| `floor_category` | `storey_range` | Extract lower floor from `"10 TO 12"` → `10`; bucket as Low (1–5), Mid (6–12), High (13+) |
| `year` | `month` | Integer year — used for stratification only, **not** a model feature |

**Target variable:** `log_resale_price_real` — using RPI-adjusted prices means the model captures structural flat-level pricing drivers rather than market-wide appreciation, since `year` is excluded as a feature.

Why log transformation? HDB resale prices have a heavy right tail that violates OLS normality assumptions. Applying `np.log()` compresses this tail, producing more normally distributed residuals. Coefficients in log space represent approximate percentage changes in price per unit increase in each feature, rather than dollar changes. Predictions are exponentiated back to SGD at evaluation — this gives the median predicted price, which is more robust to extreme transactions than the mean.

In [2]:
import re

def parse_remaining_lease(s):
    match = re.match(r"(\d+) years?(?: (\d+) months?)?", str(s))
    if match:
        years = int(match.group(1))
        months = int(match.group(2)) if match.group(2) else 0
        return round(years + months / 12, 2)
    return np.nan

df["remaining_lease_years"] = df["remaining_lease"].apply(parse_remaining_lease)

# Extract lower floor number from storey_range (e.g., "10 TO 12" → 10)
df["floor_lower"] = df["storey_range"].str.extract(r"^(\d+)").astype(int)
df["floor_category"] = pd.cut(
    df["floor_lower"],
    bins=[0, 5, 12, float("inf")],
    labels=["Low", "Mid", "High"],
    right=True
)

# Year for stratification only — will NOT be used as a model feature
df["year"] = pd.to_datetime(df["month"]).dt.year

# Target variable: log_resale_price_real
# RPI-adjusted prices are used so the model learns purely from flat characteristics
# rather than market-wide price appreciation over time, since year is not a model feature.
target = "log_resale_price_real"

print(df[["remaining_lease", "remaining_lease_years", "storey_range",
          "floor_lower", "floor_category", "year"]].head(10))

          remaining_lease  remaining_lease_years storey_range  floor_lower  \
23333   64 years 01 month                  64.08     01 TO 03            1   
23334   64 years 01 month                  64.08     07 TO 09            7   
23335            59 years                  59.00     04 TO 06            4   
23336  58 years 02 months                  58.17     04 TO 06            4   
23337   58 years 01 month                  58.08     01 TO 03            1   
23338  64 years 02 months                  64.17     07 TO 09            7   
23339  58 years 02 months                  58.17     04 TO 06            4   
23340   59 years 01 month                  59.08     04 TO 06            4   
23341            64 years                  64.00     07 TO 09            7   
23342  54 years 04 months                  54.33     04 TO 06            4   

      floor_category  year  
23333            Low  2021  
23334            Mid  2021  
23335            Low  2021  
23336            Low  202

## Step 3: Stratified Train/Test Split (80/20)

**80/20 ratio:** With ~97,000 transactions, yields ~77,000 training and ~19,000 test rows. The training set is large enough for stable OLS estimation; the test set is large enough for statistically reliable RMSE and linlin loss estimates. A 70/30 split is only preferred for datasets of a few thousand rows.

**Stratification key: `town + flat_type + year`** — three reasons:
1. **Town** — 26 HDB towns with substantially different price levels; random sampling could over-represent expensive towns in training.
2. **Flat type** — Highly imbalanced (4 ROOM ~42% vs 2 ROOM ~2%); stratifying prevents minority types being underrepresented.
3. **Year** — Despite RPI adjustment, residual structural differences exist across years (post-COVID demand surge, cooling measures); ensures proportional representation of every year in both sets.

In [3]:
from sklearn.model_selection import train_test_split

df["strat_key"] = (df["town"].astype(str) + "_" +
                   df["flat_type"].astype(str) + "_" +
                   df["year"].astype(str))

strat_counts = df["strat_key"].value_counts()
valid_keys = strat_counts[strat_counts >= 2].index
n_before = len(df)
df = df[df["strat_key"].isin(valid_keys)]
print(f"Dropped {n_before - len(df)} rows with singleton strat_key combinations. Remaining: {len(df):,}")

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["strat_key"])
print(f"Train size: {len(train_df):,} | Test size: {len(test_df):,}")

print("\nYear distribution (%):")
train_year = train_df["year"].value_counts(normalize=True).sort_index().rename("Train %")
test_year = test_df["year"].value_counts(normalize=True).sort_index().rename("Test %")
year_dist = pd.concat([train_year, test_year], axis=1)
print(year_dist.map(lambda x: f"{x:.2%}"))

print("\nFlat type distribution (%):")
train_flat = train_df["flat_type"].value_counts(normalize=True).rename("Train %")
test_flat = test_df["flat_type"].value_counts(normalize=True).rename("Test %")
flat_dist = pd.concat([train_flat, test_flat], axis=1)
print(flat_dist.map(lambda x: f"{x:.2%}"))

Dropped 7 rows with singleton strat_key combinations. Remaining: 96,789
Train size: 77,431 | Test size: 19,358

Year distribution (%):
     Train %  Test %
year                
2021  21.92%  21.91%
2022  19.92%  19.99%
2023  18.78%  18.77%
2024  20.62%  20.61%
2025  18.76%  18.73%

Flat type distribution (%):
          Train %  Test %
flat_type                
4 ROOM     41.87%  41.89%
3 ROOM     25.56%  25.54%
5 ROOM     23.59%  23.59%
EXECUTIVE   6.75%   6.77%
2 ROOM      2.22%   2.21%


## Step 4: Fit OLS Model

**Feature matrix (40 features total):**

| Type | Features | Treatment |
|------|----------|-----------|
| Continuous (9) | `remaining_lease_years`, `nearest_train_dist_m`, `dist_nearest_hawker_m`, `dist_nearest_primary_m`, `num_primary_1km`, `dist_nearest_park_m`, `dist_nearest_sportsg_m`, `dist_nearest_mall_m`, `dist_nearest_healthcare_m` | StandardScaler fit on train only; coefficients in SD units |
| Categorical — flat type (4) | 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE | One-hot; reference = 2 ROOM |
| Categorical — town (25) | All towns except ANG MO KIO | One-hot; reference = ANG MO KIO |
| Categorical — floor (2) | Mid, High | One-hot; reference = Low (floors 1–5) |

**Excluded features:**
- `floor_area_sqm` — not a user-facing input at inference time; users select flat type but do not enter exact sqm. Flat type dummies implicitly capture size.
- `dist_cbd_m` — VIF = 33.1; the 26 town fixed effects already encode CBD proximity implicitly.

In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# floor_area_sqm excluded: not a user-facing input at inference time (users select flat type,
# not exact sqm). Including a feature unavailable at prediction time makes the model undeployable.
#
# dist_cbd_m excluded: VIF = 33.1 (highest in the model), indicating severe multicollinearity
# with the 26 town dummies which already encode CBD proximity implicitly.
#
# num_primary_1km: count of MOE primary schools within 1 km radius (derived from primary_schools_1km).
# Captures school density independently of nearest-school distance — relevant because Singapore's
# Phase 2C primary school balloting prioritises proximity, making higher school counts a distinct
# pricing signal beyond just distance to the nearest school.
continuous_features = [
    "remaining_lease_years", "nearest_train_dist_m",
    "dist_nearest_hawker_m", "dist_nearest_primary_m", "num_primary_1km", "dist_nearest_park_m",
    "dist_nearest_sportsg_m", "dist_nearest_mall_m", "dist_nearest_healthcare_m"
]
categorical_features = ["flat_type", "town", "floor_category"]
reference_categories = {"flat_type": "2 ROOM", "town": "ANG MO KIO", "floor_category": "Low"}

# One-hot encode (drop_first=False), then manually drop reference categories
train_encoded = pd.get_dummies(train_df, columns=categorical_features, drop_first=False)
test_encoded = pd.get_dummies(test_df, columns=categorical_features, drop_first=False)

ref_cols = [f"{col}_{ref}" for col, ref in reference_categories.items()]
print(f"Dropping reference columns: {ref_cols}")
train_encoded = train_encoded.drop(columns=ref_cols, errors="ignore")
test_encoded = test_encoded.drop(columns=ref_cols, errors="ignore")

# Identify dummy columns
dummy_cols = [c for c in train_encoded.columns
              if any(c.startswith(f"{f}_") for f in categorical_features)]

# Standardise continuous features (fit on train only).
# Remaining lease is in years (0–99), distance features in metres (0–20,000) —
# standardising improves numerical stability of OLS.
# Dummy variables are kept as 0/1 and not standardised.
# Note: coefficients for continuous features will be in standard deviation units.
scaler = StandardScaler()
train_cont = pd.DataFrame(scaler.fit_transform(train_encoded[continuous_features]),
                           columns=continuous_features, index=train_encoded.index)
test_cont = pd.DataFrame(scaler.transform(test_encoded[continuous_features]),
                          columns=continuous_features, index=test_encoded.index)

# Cast bool dummy columns to int (pd.get_dummies returns bool in pandas 2.x)
X_train = pd.concat([train_cont, train_encoded[dummy_cols].astype(int)], axis=1)
X_test = pd.concat([test_cont, test_encoded[dummy_cols].astype(int)], axis=1)
# Align test columns to train (fill any missing with 0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
y_train = train_df[target]  # Log-space targets for model training
y_test = test_df[target]
y_train_actual = train_df['resale_price_real']  # Actual SGD prices retained for metric computation and diagnostics
y_test_actual = test_df['resale_price_real']

# VIF check
vif_df = pd.DataFrame({
    "feature": X_train.columns,
    "VIF": [variance_inflation_factor(X_train.values.astype(float), i)
            for i in range(X_train.shape[1])]
}).sort_values("VIF", ascending=False).reset_index(drop=True)
high_vif = vif_df[vif_df["VIF"] > 10]
if len(high_vif) > 0:
    print("WARNING: The following features have VIF > 10. These features show high multicollinearity "
          "and their individual coefficients may be unreliable. Flagged for manual review.")
    print(high_vif.to_string(index=False))
else:
    print("VIF check passed: No features with VIF > 10.")

# Fit OLS
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)
ols_model = sm.OLS(y_train, X_train_const).fit()
print(ols_model.summary())

## Step 5: Diagnostics

Four diagnostic checks — **for documentation purposes only**. The model is not refit or modified based on results.

| Diagnostic | What to look for |
|-----------|-----------------|
| Residuals vs Fitted | Funnel shape → heteroskedasticity; curve → non-linearity |
| Q-Q plot | Heavy tails → non-normal errors |
| Breusch-Pagan test | Formal heteroskedasticity test (p < 0.05 = detected) |
| Residuals by segment | \|mean residual\| > $30,000 → systematic over/underprediction |

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats
from statsmodels.stats.diagnostic import het_breuschpagan

y_train_pred = ols_model.predict(X_train_const)

# Log-space residuals — used for QQ plot, Breusch-Pagan test, residuals vs fitted
residuals_log = y_train - y_train_pred
std_resid = (residuals_log - residuals_log.mean()) / residuals_log.std()

# Actual price residuals — used for town and flat type systematic error breakdown
residuals_actual = y_train_actual - np.exp(y_train_pred)

# 1. Residual vs Fitted plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_train_pred, residuals_log, alpha=0.3, s=5)
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Fitted Values")
ax.set_ylabel("Residuals")
ax.set_title("Residuals vs Fitted Values")
plt.tight_layout()
plt.show()
print("NOTE: Residuals and fitted values are in log space. Inspect for funnel shape (heteroskedasticity) or curve (non-linearity).")

# 2. Q-Q plot
fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(std_resid, dist="norm", plot=ax)
ax.set_title("Q-Q Plot of Standardised Residuals")
plt.tight_layout()
plt.show()
print("NOTE: Residuals are in log space — compare to previous QQ plot to verify normality has improved.")

# 3. Breusch-Pagan test
bp_lm, bp_pvalue, _, _ = het_breuschpagan(residuals_log, X_train_const)
print(f"\nBreusch-Pagan test: LM stat = {bp_lm:.4f}, p-value = {bp_pvalue:.4e}")
if bp_pvalue < 0.05:
    print("WARNING: Heteroskedasticity detected (p < 0.05). Standard errors may be unreliable.")
else:
    print("Breusch-Pagan: No significant heteroskedasticity (p >= 0.05).")

# 4. Residuals by town and flat type
resid_df = pd.DataFrame({
    "town": train_df["town"].values,
    "flat_type": train_df["flat_type"].values,
    "residual": residuals_actual.values
})
print("\nMean residual by town (sorted):")
town_resid = resid_df.groupby("town")["residual"].mean().sort_values()
print(town_resid.to_string())
flagged_towns = town_resid[town_resid.abs() > 30000]
if len(flagged_towns) > 0:
    print(f"\nWARNING: Towns with |mean residual| > $30,000 (systematic over/underprediction):")
    print(flagged_towns.to_string())

print("\nMean residual by flat_type (sorted):")
flat_resid = resid_df.groupby("flat_type")["residual"].mean().sort_values()
print(flat_resid.to_string())
flagged_flats = flat_resid[flat_resid.abs() > 30000]
if len(flagged_flats) > 0:
    print(f"\nWARNING: Flat types with |mean residual| > $30,000 (systematic over/underprediction):")
    print(flagged_flats.to_string())

## Step 6: Evaluation

**Metrics computed on the held-out test set (19,358 transactions):**

| Metric | Description |
|--------|-------------|
| **RMSE** | Root Mean Squared Error — penalises large errors quadratically |
| **Linlin Loss** | Asymmetric linear loss (underprediction weight = 2.0). Underprediction penalised 2× because underestimating fair value increases the buyer's risk of paying Cash Over Valuation (COV) during negotiation. |

**Prediction intervals:** 80% intervals (10th–90th percentile) via `get_prediction().summary_frame(alpha=0.2)`. Coverage should be close to 80% for a well-calibrated model.

> Note: A composite 50/50 score across RMSE and linlin loss will be computed in a final model comparison notebook once all candidate models are evaluated.

In [6]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    """
    Asymmetric linear loss penalising underprediction more heavily than overprediction.
    underpredict_weight=2.0 means predicted < actual is penalised at 2x.
    Overprediction is penalised at 1x.
    Rationale: Underestimating fair value increases the buyer's risk of paying
    Cash Over Valuation (COV) when negotiating price.
    """
    errors = np.array(y_true) - np.array(y_pred)  # positive = underprediction
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

y_test_pred_log = ols_model.predict(X_test_const)
y_test_pred = np.exp(y_test_pred_log)  # median prediction — no Duan correction applied
test_rmse = rmse(y_test_actual, y_test_pred)
test_linlin = linlin_loss(y_test_actual, y_test_pred, underpredict_weight=2.0)

print("=== OLS MODEL EVALUATION ===")
print(f"RMSE:          ${test_rmse:,.0f}")
print(f"Linlin Loss:   ${test_linlin:,.0f} (underpredict weight = 2.0)")
print()
print("NOTE: Composite 50/50 scoring across models will be computed in the")
print("final comparison notebook once all models are evaluated.")

# 80% prediction intervals (alpha=0.2 → 10th–90th percentile)
pred_frame = ols_model.get_prediction(X_test_const).summary_frame(alpha=0.2)
pi_lower = np.exp(pred_frame['obs_ci_lower'].values)
pi_upper = np.exp(pred_frame['obs_ci_upper'].values)
coverage = ((y_test_actual.values >= pi_lower) & (y_test_actual.values <= pi_upper)).mean()
print(f"\n80% Prediction Interval Coverage: {coverage:.1%}")
# Coverage should be close to 80% for a well-calibrated model.
if coverage < 0.70:
    print("WARNING: Coverage < 70%. Prediction intervals are too narrow — model may be overconfident.")
elif coverage > 0.90:
    print("WARNING: Coverage > 90%. Prediction intervals are too wide — model may be underconfident.")
else:
    print("Coverage within expected range (70%–90%).")

=== OLS MODEL EVALUATION ===
RMSE:          $76,319
Linlin Loss:   $83,305 (underpredict weight = 2.0)

NOTE: Composite 50/50 scoring across models will be computed in the
final comparison notebook once all models are evaluated.

80% Prediction Interval Coverage: 82.8%
Coverage within expected range (70%–90%).


## Results Summary

| Metric | Value |
|--------|-------|
| R² (train, log space) | 0.887 |
| Adj. R² (train, log space) | 0.887 |
| RMSE (test, SGD) | $76,377 |
| Linlin Loss (test, SGD, w=2) | $83,218 |
| 80% PI Coverage | 82.7% ✓ (within 70–90% expected range) |

**VIF check:** All 39 features passed (no VIF > 10) after excluding `floor_area_sqm` and `dist_cbd_m`.

**Diagnostics:**
- Breusch-Pagan: heteroskedasticity expected (housing price variance tends to be larger for expensive flats)
- Residuals by town and flat type: mean training residuals are ~0 by construction in OLS